# **1. Finetune**

In [ ]:
# Cell 1 - Cài đặt thư viện
!pip install -q --upgrade transformers trl==0.7.0
!pip install -q accelerate bitsandbytes peft datasets
!pip install -q sentencepiece

print("Đã cài đặt xong các thư viện")

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files="/kaggle/input/datafull/file_tong_hop.jsonl")
print(data)
print(data["train"][0])

In [ ]:
from transformers import AutoTokenizer

#  Chọn model Qwen để huấn luyện (dung lượng nhỏ hơn nếu RAM thấp)
# Gợi ý: 
# - "Qwen/Qwen2.5-0.5B-Instruct"  → nhẹ, phù hợp Kaggle free
# - "Qwen/Qwen2.5-1.5B-Instruct" → mạnh hơn, cần nhiều RAM hơn
model_id = "Qwen/Qwen3-4B"

#  Tải tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

#  Hàm định dạng dữ liệu
def format_chat(example):
    """
    Chuyển danh sách messages trong mỗi mẫu dữ liệu 
    thành text hoàn chỉnh theo template chat của Qwen.
    """
    text = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return {"text": text}

#  Map dữ liệu qua hàm format_chat
train_dataset = data["train"].map(format_chat)

#  Kiểm tra kết quả
print(train_dataset[0])

In [ ]:
# Thêm ở đầu file
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# === Fine-tune using transformers.Trainer + PEFT ===
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
import torch
import os

# Suppress specific warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# --- Configurable ---
model_id = "Qwen/Qwen3-4B"
output_dir = "/kaggle/working/qwen_finetuned"
max_length = 512
batch_size = 1
grad_accum = 4
num_epochs = 1
learning_rate = 2e-4

print(" Đang khởi tạo fine-tuning Qwen3-4B...")

# --- Load tokenizer & model ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model với cấu hình tối ưu
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(" Đã tải model và tokenizer")

# --- Tokenize dataset ---
def tokenize_fn(batch):
    out = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=train_dataset.column_names)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f" Đã tokenize {len(tokenized)} samples")

# --- Data collator ---
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- PEFT (LoRA) config ---
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap model với LoRA
model = get_peft_model(model, peft_config)
print(" Đã áp dụng LoRA")

# --- TrainingArguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    learning_rate=learning_rate,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],  # Tắt logging để giảm warnings
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(" Bắt đầu training...")

# --- Train ---
try:
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(" Fine-tune thành công! Model đã lưu tại:", output_dir)
except RuntimeError as e:
    print(" Lỗi trong quá trình training:", str(e))
    if "out of memory" in str(e).lower():
        print("Gợi ý: Giảm max_length xuống 256 hoặc batch_size")
except Exception as e:
    print(" Lỗi không xác định:", str(e))

In [ ]:
trainer.save_model("/kaggle/working/qwen_finetuned")
tokenizer.save_pretrained("/kaggle/working/qwen_finetuned")